# Watershed Dashboard Tutorial
## Working with ArcGIS Shapefiles and Creating Interactive Visualizations

This notebook demonstrates how to:
1. Load and explore watershed shapefiles
2. Perform spatial analysis
3. Create visualizations
4. Prepare data for the dashboard

In [ ]:
# Import required libraries
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from pathlib import Path

print("✅ Libraries imported successfully")

## 1. Create Sample Data

First, let's create sample watershed data to work with:

In [ ]:
from create_sample_data import create_sample_watersheds

# Create sample data
gdf = create_sample_watersheds(output_path='sample_data/watersheds.shp', num_watersheds=15)

print(f"Created {len(gdf)} watersheds")
print(f"Attributes: {list(gdf.columns)}")

## 2. Load and Explore Shapefile

Load your shapefile and examine its properties:

In [ ]:
# Load shapefile
shapefile_path = 'sample_data/watersheds.shp'
gdf = gpd.read_file(shapefile_path)

# Display basic information
print(f"Number of features: {len(gdf)}")
print(f"Coordinate System: {gdf.crs}")
print(f"\nColumns: {list(gdf.columns)}")
print(f"\nGeometry types: {gdf.geometry.geom_type.unique()}")

In [ ]:
# Display first few rows
gdf.head()

In [ ]:
# Get summary statistics
gdf.describe()

## 3. Spatial Analysis

Perform basic spatial operations:

In [ ]:
# Calculate total area
total_area = gdf['AREA_SQKM'].sum()
print(f"Total watershed area: {total_area:.2f} km²")

# Calculate centroids
gdf['centroid'] = gdf.geometry.centroid

# Calculate perimeter-to-area ratio
gdf['shape_index'] = gdf['PERIMETER_KM'] / np.sqrt(gdf['AREA_SQKM'])

print(f"\nShape complexity index range: {gdf['shape_index'].min():.2f} - {gdf['shape_index'].max():.2f}")

In [ ]:
# Find neighboring watersheds
gdf['neighbors'] = None

for idx, row in gdf.iterrows():
    neighbors = gdf[gdf.geometry.touches(row.geometry)].index.tolist()
    gdf.at[idx, 'neighbors'] = len(neighbors)

print(f"Average neighbors per watershed: {gdf['neighbors'].mean():.1f}")

## 4. Create Visualizations

Generate various plots and maps:

In [ ]:
# Plot watersheds
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Simple boundary plot
gdf.plot(ax=axes[0], color='lightblue', edgecolor='black', linewidth=1)
axes[0].set_title('Watershed Boundaries')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

# Choropleth by area
gdf.plot(column='AREA_SQKM', ax=axes[1], legend=True, 
         cmap='YlOrRd', edgecolor='black', linewidth=0.5)
axes[1].set_title('Watersheds by Area')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution histograms
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

gdf['AREA_SQKM'].hist(bins=20, ax=axes[0, 0], color='steelblue', edgecolor='black')
axes[0, 0].set_title('Distribution of Watershed Area')
axes[0, 0].set_xlabel('Area (km²)')
axes[0, 0].set_ylabel('Count')

gdf['NITROGEN_KG'].hist(bins=20, ax=axes[0, 1], color='green', edgecolor='black')
axes[0, 1].set_title('Distribution of Nitrogen Load')
axes[0, 1].set_xlabel('Nitrogen (kg/yr)')
axes[0, 1].set_ylabel('Count')

gdf['FOREST_PCT'].hist(bins=20, ax=axes[1, 0], color='darkgreen', edgecolor='black')
axes[1, 0].set_title('Distribution of Forest Cover')
axes[1, 0].set_xlabel('Forest (%)')
axes[1, 0].set_ylabel('Count')

gdf['POPULATION'].hist(bins=20, ax=axes[1, 1], color='coral', edgecolor='black')
axes[1, 1].set_title('Distribution of Population')
axes[1, 1].set_xlabel('Population')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots for relationships
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(gdf['AREA_SQKM'], gdf['NITROGEN_KG'], alpha=0.6, s=100)
axes[0].set_xlabel('Watershed Area (km²)')
axes[0].set_ylabel('Nitrogen Load (kg/yr)')
axes[0].set_title('Nitrogen Load vs. Watershed Area')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(gdf['AGRICULTURE_PCT'], gdf['NITROGEN_KG'], alpha=0.6, s=100, c='green')
axes[1].set_xlabel('Agriculture (%)')
axes[1].set_ylabel('Nitrogen Load (kg/yr)')
axes[1].set_title('Nitrogen Load vs. Agricultural Land')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Create Interactive Map

Generate an interactive Folium map:

In [ ]:
# Create interactive map
center_lat = gdf.geometry.centroid.y.mean()
center_lon = gdf.geometry.centroid.x.mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=8)

# Add watersheds
for idx, row in gdf.iterrows():
    # Create popup text
    popup_text = f"""
    <b>{row['NAME']}</b><br>
    ID: {row['WATERSHED_ID']}<br>
    Area: {row['AREA_SQKM']:.2f} km²<br>
    Population: {row['POPULATION']:,}<br>
    Nitrogen: {row['NITROGEN_KG']:.1f} kg/yr<br>
    Forest: {row['FOREST_PCT']:.1f}%
    """
    
    folium.GeoJson(
        row.geometry,
        style_function=lambda x: {
            'fillColor': '#3388ff',
            'color': '#000000',
            'weight': 2,
            'fillOpacity': 0.5
        },
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

m

## 6. Advanced Analysis

Perform more complex watershed analysis:

In [ ]:
# Calculate nutrient yields (per unit area)
gdf['nitrogen_yield'] = gdf['NITROGEN_KG'] / gdf['AREA_SQKM']
gdf['phosphorus_yield'] = gdf['PHOSPHORUS_KG'] / gdf['AREA_SQKM']

# Calculate N:P ratio
gdf['np_ratio'] = gdf['NITROGEN_KG'] / gdf['PHOSPHORUS_KG']

print("Nutrient Yield Statistics:")
print(f"Nitrogen yield range: {gdf['nitrogen_yield'].min():.1f} - {gdf['nitrogen_yield'].max():.1f} kg/km²/yr")
print(f"Phosphorus yield range: {gdf['phosphorus_yield'].min():.1f} - {gdf['phosphorus_yield'].max():.1f} kg/km²/yr")
print(f"Average N:P ratio: {gdf['np_ratio'].mean():.2f}")

In [ ]:
# Classify watersheds by water quality risk
def classify_risk(row):
    score = 0
    
    # High nitrogen yield
    if row['nitrogen_yield'] > gdf['nitrogen_yield'].quantile(0.75):
        score += 2
    
    # High urban percentage
    if row['URBAN_PCT'] > 20:
        score += 1
    
    # Low forest cover
    if row['FOREST_PCT'] < 30:
        score += 1
    
    # High agriculture
    if row['AGRICULTURE_PCT'] > 40:
        score += 1
    
    if score >= 4:
        return 'High Risk'
    elif score >= 2:
        return 'Medium Risk'
    else:
        return 'Low Risk'

gdf['risk_category'] = gdf.apply(classify_risk, axis=1)

# Count by category
print("\nWater Quality Risk Classification:")
print(gdf['risk_category'].value_counts())

In [ ]:
# Plot risk categories
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

color_map = {'Low Risk': 'green', 'Medium Risk': 'yellow', 'High Risk': 'red'}
gdf.plot(column='risk_category', ax=ax, legend=True, 
         categorical=True, cmap='RdYlGn_r', edgecolor='black', linewidth=1)

ax.set_title('Watershed Water Quality Risk Assessment', fontsize=16)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.tight_layout()
plt.show()

## 7. Export Results

Save your analysis results:

In [ ]:
# Export to CSV (without geometry)
output_df = gdf.drop(columns=['geometry', 'centroid'])
output_df.to_csv('sample_data/watershed_analysis.csv', index=False)
print("✅ Analysis exported to CSV")

# Export to GeoJSON
gdf.to_file('sample_data/watersheds_analyzed.geojson', driver='GeoJSON')
print("✅ Analysis exported to GeoJSON")

# Export to new shapefile
gdf.to_file('sample_data/watersheds_analyzed.shp')
print("✅ Analysis exported to Shapefile")

## 8. Prepare for Dashboard

Validate and prepare data for the dashboard:

In [ ]:
from shapefile_utils import validate_shapefile, get_shapefile_info

# Validate the shapefile
report = validate_shapefile('sample_data/watersheds.shp')

print("Validation Report:")
print(f"Valid: {report['valid']}")
print(f"\nWarnings: {len(report['warnings'])}")
for warning in report['warnings']:
    print(f"  - {warning}")

print(f"\nErrors: {len(report['errors'])}")
for error in report['errors']:
    print(f"  - {error}")

In [ ]:
# Get detailed information
info = get_shapefile_info('sample_data/watersheds.shp', detailed=True)

print("\nDataset Information:")
print(f"Features: {info['feature_count']}")
print(f"Columns: {info['columns']}")
print(f"CRS: {info['crs']}")
print(f"\nNumeric Attributes:")

for col, col_info in info['column_info'].items():
    if 'mean' in col_info:
        print(f"\n{col}:")
        print(f"  Range: {col_info['min']:.2f} - {col_info['max']:.2f}")
        print(f"  Mean: {col_info['mean']:.2f}")
        print(f"  Median: {col_info['median']:.2f}")

## 9. Summary Statistics for Dashboard

Generate a summary report:

In [ ]:
summary = f"""
WATERSHED ANALYSIS SUMMARY
{'='*50}

Dataset Overview:
  - Total Watersheds: {len(gdf)}
  - Total Area: {gdf['AREA_SQKM'].sum():.2f} km²
  - Total Population: {gdf['POPULATION'].sum():,}

Land Use Summary:
  - Average Forest Cover: {gdf['FOREST_PCT'].mean():.1f}%
  - Average Agriculture: {gdf['AGRICULTURE_PCT'].mean():.1f}%
  - Average Urban: {gdf['URBAN_PCT'].mean():.1f}%

Nutrient Loads:
  - Total Nitrogen: {gdf['NITROGEN_KG'].sum():.0f} kg/yr
  - Total Phosphorus: {gdf['PHOSPHORUS_KG'].sum():.0f} kg/yr
  - Average N:P Ratio: {gdf['np_ratio'].mean():.2f}

Risk Assessment:
  - High Risk Watersheds: {(gdf['risk_category'] == 'High Risk').sum()}
  - Medium Risk Watersheds: {(gdf['risk_category'] == 'Medium Risk').sum()}
  - Low Risk Watersheds: {(gdf['risk_category'] == 'Low Risk').sum()}

{'='*50}
"""

print(summary)

# Save summary to file
with open('sample_data/analysis_summary.txt', 'w') as f:
    f.write(summary)

print("\n✅ Summary saved to file")

## Next Steps

Now you can:
1. Run the dashboard: `streamlit run watershed_dashboard.py`
2. Load your analyzed shapefile in the dashboard
3. Explore the interactive visualizations
4. Share the dashboard with stakeholders

The dashboard provides:
- Interactive maps with multiple base layers
- Statistical summaries and charts
- Data filtering and export capabilities
- Professional presentation of your analysis